# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [ ]:
# Uses model output from w05_model.ipynb (best model, test-set scores)
# and the honest-split status from w06_validation_audit.ipynb.

import pandas as pd

# Re-score test set with best model from w05 (Random Forest or Logistic Regression)
queue = X_test.copy()
queue["client_id"] = df.iloc[test_idx]["client_id"].values
queue["page_id"] = df.iloc[test_idx]["page_id"].values
queue["decline_score"] = probs  # from w05's best_pipe.predict_proba

def reason_code(row):
    codes = []
    if row.get("days_since_update", 0) > 270:
        codes.append("STALE_270D")  # matches paper Finding #2 decay window
    if row.get("avg_position", 99) > 20:
        codes.append("LOW_RANK")
    if row.get("word_count", 0) < 1000:
        codes.append("THIN_CONTENT")
    if not codes:
        codes.append("MODEL_FLAGGED_OTHER")
    return codes

queue["reason_codes"] = queue.apply(reason_code, axis=1)
queue = queue.sort_values("decline_score", ascending=False)
queue.head(20)

Actions are ranked by decline_score (the model's probability output), and each gets one or more reason codes so a human reviewer sees why a page was flagged, not just a number. Reason codes are simple, auditable rules (stale, low-rank, thin) rather than opaque — a reviewer can check them by eye without trusting the model blindly.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Intended use: decision-support for prioritizing which pages a content editor reviews first for a refresh. The queue orders candidates; it does not decide what changes to make or execute any change.

Limits:

Trained and evaluated on a client-holdout split — performance on entirely new clients is directional, not guaranteed (see w06, section 2 gap: time-aware split not yet re-verified).
is_declining_label is a proxy (trend_direction == "down"), not a ground-truth signal of business harm — a page can decline in impressions for reasons unrelated to content quality (seasonality, SERP changes).
The model has never seen a causal refresh outcome — it flags candidates, it does not predict that refreshing will fix anything (same caveat the paper itself gives for Finding #4).
Feature set audit is incomplete (w06 section 3's leakage re-run is a documented TODO) — treat scores as provisional until that's closed.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Every action in this queue requires human review before execution. Nothing here auto-publishes.

No-go — do NOT automate:

Auto-publishing rewritten content without editor sign-off
Auto-merging/redirecting "cannibalizing" pages without checking business intent (a client may want two similar pages for different funnel stages)
Treating decline_score as a performance-guarantee number in client-facing reporting
Acting on any page with MODEL_FLAGGED_OTHER as the only reason code without a manual look — that code means the rules didn't explain the flag, which is exactly when human judgment matters mos

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [ ]:
retrain_triggers = {
    "data_drift": "Re-check honest-split precision@50 monthly; retrain if it drops >15% from this run's baseline",
    "label_definition_change": "If trend_direction's calculation window or threshold changes upstream, retrain — the label itself shifted",
    "new_client_cohort": "If >20% of queue clients are unseen-at-training clients, re-verify cross-brand accuracy (paper's own cross-brand vs same-brand gap, e.g. 90% vs 75%, is the pattern to watch for)",
    "leakage_audit_reopened": "Any new engineered feature added to refresh_feature_vector.csv triggers a re-run of the w06 leakage hunt before it's trusted in the queue",
}

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
import json, os
os.makedirs("../outputs", exist_ok=True)
os.makedirs("../figures", exist_ok=True)

queue.to_csv("../outputs/ranked_action_queue.csv", index=False)  # gitignored by CI design

metrics = {
    "model": best_name,
    "precision_at_50": float(results_df.loc[results_df.model == best_name, "precision@50"].iloc[0]),
    "baseline_precision_at_50": float(results_df.loc[results_df.model == "Baseline (hand-rule)", "precision@50"].iloc[0]),
    "split": "client-grouped (GroupShuffleSplit)",
    "known_gaps": ["time-aware split not re-verified", "leakage AUC re-run pending"],
}
with open("../outputs/w07_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(metrics)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.